# MuSGD × YOLO26: pilot experiment log

Research question: how does the MuSGD/AdamW fine-tuning comparison change with scarce images and missing training boxes?

This notebook runs **8 development runs** with provisional learning rates. It evaluates clean **validation only**. Test data are audited for integrity but never used for configuration selection or scoring here. No paper-level performance claim follows from this run.

Enable a Kaggle GPU and Internet. Source code, tests and configuration are embedded below, with a SHA-256 integrity check. Source images are downloaded from the official dataset location and are not redistributed in this notebook.

The frozen split policy preserves test, excludes exact overlaps from validation/training, and excludes 12 visually reviewed training near duplicates. All retained labels are unchanged; source membership differs from the official splits. Inspect `split_audit.json` and `configs/duplicate_exclusions.json`.


## 1. Install the fixed training library
Kaggle's existing CUDA-compatible PyTorch is retained; the complete environment is saved after import.

In [ ]:
%pip install -q "ultralytics==8.4.187" "PyYAML==6.0.3" "matplotlib>=3.8,<4" "Pillow>=10" "pytest>=8,<10"

## 2. Restore the experiment source
The bundle contains src/, tests/, and configs/. Inspect the restored files before changing a study.

In [ ]:
from pathlib import Path
import base64, hashlib, io, os, sys, zipfile

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "work"
PROJECT = WORKING / "muon_yolo_project"
PROJECT.mkdir(parents=True, exist_ok=True)
bundle = base64.b64decode('...')
assert hashlib.sha256(bundle).hexdigest() == '3202ea98ae493c0e07e9d3f6bc41ef6f441cc537ae503ffbd1cab3a8ccc656dd'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        destination = (PROJECT / member).resolve()
        assert destination.is_relative_to(PROJECT.resolve())
    archive.extractall(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Restored source:", PROJECT)


## 3. Verify corruption and leakage safeguards
These tests use synthetic fixture files, not scientific training data.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q", str(PROJECT / "tests")], cwd=PROJECT, check=True)


## 4. Review the bounded run plan
The smoke mode uses 5% of training images, 20% per-object missing boxes, two epochs, and two explicit optimizers. Validation remains full size. All runs start from identical checkpoint bytes and the initialized parameters are checked for pairing.

In [ ]:
import json
from src.experiment import load_config, planned_runs, run_study
PROFILE = 'pilot'
CONFIG = PROJECT / "configs" / f"{PROFILE}.yaml"
OUTPUT = WORKING / f"muon_yolo_{PROFILE}"
config = load_config(CONFIG)
print(json.dumps(config, indent=2))
print("Planned runs:", planned_runs(config))


## 5. Audit data, train, and evaluate clean validation
Outputs include SHA-256 audits, corruption examples, checkpoint lineage, actual optimizer groups, initialized-model hashes, optimizer step counts, histories, timings, peak training VRAM and failure records. A changed code/config/environment requires a new output directory; failed runs get a new attempt directory.

In [ ]:
rows = run_study(CONFIG, OUTPUT, max_runs=8)


## 6. Inspect measured development results
Single-seed smoke scores verify plumbing. They do not rank optimizers or establish a scarcity/noise interaction.

In [ ]:
import pandas as pd
from IPython.display import display, Image
results = pd.read_csv(OUTPUT / "results.csv")
display(results[["optimizer", "status", "train_images", "realized_noise_rate", "val_map50_95", "training_seconds", "optimizer_steps"]])
display(Image(filename=str(OUTPUT / "validation_map.png")))
display(Image(filename=str(OUTPUT / "training_time.png")))


## 7. Retrieve evidence and plan the paper
Download the notebook outputs through Kaggle or `kaggle kernels output OWNER/SLUG -p outputs/PROFILE`. Keep results.csv, manifest.json, source_audit.json, variant_audit.json, checkpoint_lineage.json, environment-lock.txt and run histories. Preserve all tried settings. Before a final paper: equal validation tuning budgets, SGD baseline, repeated paired seeds and corruption seeds, a second public dataset, and a frozen one-shot test protocol. Dataset licensing and possible pretrained-image overlap remain unresolved.